# View Lineage & Change-History Tracker — Demo Setup

This notebook is a **self-contained demo**. It:

1. Creates a sample schema, sample tables, and sample views (with aggregations/joins).
2. "Evolves" the views over time (simulating real changes a client would make).
3. Extracts view metadata, source-table lineage, and change history.
4. Writes everything to a single **Excel file** with multiple sheets.

**Assumptions** (change in the widgets below if different):
- Your workspace has **Unity Catalog** enabled.
- Catalog = `main`, Schema = `view_lineage_demo` (you can change these via widgets).
- If Unity Catalog is *not* enabled, the notebook falls back to the default `hive_metastore`
  catalog automatically — lineage from `information_schema.view_table_usage` will not be
  available in that fallback mode (it's a Unity Catalog feature), but everything else still works.

> Real "change history" for a view's definition is not natively versioned by Databricks the way
> Delta tables are (`DESCRIBE HISTORY` only works on Delta tables, not views). So this notebook
> demonstrates **two** approaches side-by-side:
> - A **custom change-log table** (`view_change_log`) that we populate ourselves every time we
>   `CREATE OR REPLACE VIEW` — this is what you'd wire into a real client pipeline (e.g. via a
>   deployment job / CI step that logs each view change).
> - An **optional read from `system.access.audit`** (Unity Catalog system tables), which contains
>   the *real* historical `createView` / `alterView` events — but this requires an admin to have
>   enabled system tables and requires you to have `SELECT` on `system.access.audit`. The notebook
>   tries this and gracefully skips it if unavailable, so the notebook never breaks.

In [0]:
%pip install -q openpyxl
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Configuration

In [0]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "view_lineage_demo")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

USE_UC = True
try:
    spark.sql(f"USE CATALOG {CATALOG}")
except Exception as e:
    print(f"Could not USE CATALOG {CATALOG} ({e}). Falling back to workspace catalog.")
    CATALOG = "workspace"
    USE_UC = True
    spark.sql(f"USE CATALOG {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Using catalog='{CATALOG}', schema='{SCHEMA}', Unity Catalog={USE_UC}")

Could not USE CATALOG main ([NO_SUCH_CATALOG_EXCEPTION] Catalog 'main' was not found. Please verify the catalog name and then retry the query or command again. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.NoSuchCatalogException
	at com.databricks.sql.managedcatalog.ManagedCatalogSessionCatalog.requireCtExists(ManagedCatalogSessionCatalog.scala:706)
	at com.databricks.sql.managedcatalog.ManagedCatalogSessionCatalog.setCurrentCatalog(ManagedCatalogSessionCatalog.scala:829)
	at com.databricks.sql.DatabricksCatalogManager.setCurrentCatalog(DatabricksCatalogManager.scala:219)
	at org.apache.spark.sql.execution.command.SetCatalogCommand.run(SetCatalogCommand.scala:62)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$2(commands.scala:87)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:202)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.$anonfun$sideEffectResult$1(commands.

## 2. Sample tables + sample data

In [0]:
from pyspark.sql import Row
from datetime import date

customers_data = [
    Row(customer_id=1, customer_name="Aditi Rao",     city="Chennai",   signup_date=date(2023, 1, 12)),
    Row(customer_id=2, customer_name="Rahul Mehta",    city="Mumbai",    signup_date=date(2023, 3, 5)),
    Row(customer_id=3, customer_name="Sara Thomas",    city="Bengaluru", signup_date=date(2023, 4, 20)),
    Row(customer_id=4, customer_name="Vikram Nair",    city="Chennai",   signup_date=date(2023, 6, 1)),
    Row(customer_id=5, customer_name="Priya Iyer",     city="Hyderabad", signup_date=date(2023, 7, 15)),
]

products_data = [
    Row(product_id=101, product_name="Wireless Mouse",   category="Electronics", unit_price=799.0),
    Row(product_id=102, product_name="Mechanical Keyboard", category="Electronics", unit_price=2999.0),
    Row(product_id=103, product_name="Office Chair",     category="Furniture",   unit_price=6499.0),
    Row(product_id=104, product_name="Standing Desk",    category="Furniture",   unit_price=15999.0),
    Row(product_id=105, product_name="Notebook Set",     category="Stationery",  unit_price=249.0),
]

orders_data = [
    Row(order_id=1001, customer_id=1, product_id=101, order_date=date(2024, 1, 10), quantity=2),
    Row(order_id=1002, customer_id=1, product_id=105, order_date=date(2024, 2, 3),  quantity=5),
    Row(order_id=1003, customer_id=2, product_id=102, order_date=date(2024, 2, 15), quantity=1),
    Row(order_id=1004, customer_id=3, product_id=103, order_date=date(2024, 3, 1),  quantity=1),
    Row(order_id=1005, customer_id=3, product_id=104, order_date=date(2024, 3, 1),  quantity=1),
    Row(order_id=1006, customer_id=4, product_id=101, order_date=date(2024, 4, 12), quantity=3),
    Row(order_id=1007, customer_id=5, product_id=102, order_date=date(2024, 5, 20), quantity=2),
    Row(order_id=1008, customer_id=2, product_id=105, order_date=date(2024, 6, 2),  quantity=10),
    Row(order_id=1009, customer_id=4, product_id=104, order_date=date(2024, 7, 18), quantity=1),
    Row(order_id=1010, customer_id=5, product_id=103, order_date=date(2024, 8, 9),  quantity=2),
]

spark.createDataFrame(customers_data).write.mode("overwrite").format("delta") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.customers")

spark.createDataFrame(products_data).write.mode("overwrite").format("delta") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.products")

spark.createDataFrame(orders_data).write.mode("overwrite").format("delta") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.orders")

print("Sample tables created: customers, products, orders")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

Sample tables created: customers, products, orders


database,tableName,isTemporary
view_lineage_demo,customers,false
view_lineage_demo,orders,false
view_lineage_demo,products,false
view_lineage_demo,view_change_log,false
view_lineage_demo,vw_customer_order_summary,false
view_lineage_demo,vw_product_sales_agg,false


## 3. Change-log table (tracks every view change we make)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.view_change_log (
    view_full_name   STRING,
    definition_sql   STRING,
    change_description STRING,
    changed_by       STRING,
    changed_at       TIMESTAMP
) USING DELTA
""")

def create_or_replace_view(view_name, select_sql, change_description, changed_at_override=None):
    """
    Creates/replaces a view and records the change in view_change_log.
    changed_at_override lets us simulate a historical timestamp for this demo
    (in a real pipeline you'd just use current_timestamp() — the change is logged
    at the moment the deployment/migration actually runs).
    """
    view_full_name = f"{CATALOG}.{SCHEMA}.{view_name}"
    spark.sql(f"CREATE OR REPLACE VIEW {view_full_name} AS {select_sql}")

    changed_by = spark.sql("SELECT current_user()").collect()[0][0]
    ts_expr = f"TIMESTAMP('{changed_at_override}')" if changed_at_override else "current_timestamp()"

    escaped_sql = select_sql.replace("'", "''")
    escaped_desc = change_description.replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.view_change_log
        SELECT
            '{view_full_name}' AS view_full_name,
            '{escaped_sql}'    AS definition_sql,
            '{escaped_desc}'   AS change_description,
            '{changed_by}'     AS changed_by,
            {ts_expr}          AS changed_at
    """)
    print(f"[{change_description}] -> {view_full_name}")

print("Helper `create_or_replace_view` ready.")

Helper `create_or_replace_view` ready.


## 4. Create the views — and evolve them over time

Two views are created, each going through 3 revisions to simulate real client-side changes:

- `vw_customer_order_summary` — join of orders + customers + products
- `vw_product_sales_agg` — aggregation of revenue/quantity by product & category

In [0]:
# ---------- View 1: vw_customer_order_summary ----------

create_or_replace_view(
    "vw_customer_order_summary",
    f"""
        SELECT
            o.order_id, o.order_date, c.customer_name, p.product_name, o.quantity
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.customers c ON o.customer_id = c.customer_id
        JOIN {CATALOG}.{SCHEMA}.products  p ON o.product_id  = p.product_id
    """,
    "v1: initial join of orders, customers, products",
    changed_at_override="2024-01-15 10:00:00",
)

create_or_replace_view(
    "vw_customer_order_summary",
    f"""
        SELECT
            o.order_id, o.order_date, c.customer_name, c.city,
            p.product_name, p.category, o.quantity,
            ROUND(o.quantity * p.unit_price, 2) AS total_amount
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.customers c ON o.customer_id = c.customer_id
        JOIN {CATALOG}.{SCHEMA}.products  p ON o.product_id  = p.product_id
    """,
    "v2: added customer city, product category and computed total_amount",
    changed_at_override="2024-04-02 14:30:00",
)

create_or_replace_view(
    "vw_customer_order_summary",
    f"""
        SELECT
            o.order_id, o.order_date, c.customer_name, c.city,
            p.product_name, p.category, o.quantity,
            ROUND(o.quantity * p.unit_price, 2) AS total_amount
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.customers c ON o.customer_id = c.customer_id
        JOIN {CATALOG}.{SCHEMA}.products  p ON o.product_id  = p.product_id
        WHERE o.order_date >= DATE'2024-01-01'
    """,
    "v3: added filter to restrict to orders from 2024 onward",
    changed_at_override="2024-08-20 09:15:00",
)

display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.vw_customer_order_summary ORDER BY order_id"))

[v1: initial join of orders, customers, products] -> workspace.view_lineage_demo.vw_customer_order_summary
[v2: added customer city, product category and computed total_amount] -> workspace.view_lineage_demo.vw_customer_order_summary
[v3: added filter to restrict to orders from 2024 onward] -> workspace.view_lineage_demo.vw_customer_order_summary


order_id,order_date,customer_name,city,product_name,category,quantity,total_amount
1001,2024-01-10,Aditi Rao,Chennai,Wireless Mouse,Electronics,2,1598.0
1002,2024-02-03,Aditi Rao,Chennai,Notebook Set,Stationery,5,1245.0
1003,2024-02-15,Rahul Mehta,Mumbai,Mechanical Keyboard,Electronics,1,2999.0
1004,2024-03-01,Sara Thomas,Bengaluru,Office Chair,Furniture,1,6499.0
1005,2024-03-01,Sara Thomas,Bengaluru,Standing Desk,Furniture,1,15999.0
1006,2024-04-12,Vikram Nair,Chennai,Wireless Mouse,Electronics,3,2397.0
1007,2024-05-20,Priya Iyer,Hyderabad,Mechanical Keyboard,Electronics,2,5998.0
1008,2024-06-02,Rahul Mehta,Mumbai,Notebook Set,Stationery,10,2490.0
1009,2024-07-18,Vikram Nair,Chennai,Standing Desk,Furniture,1,15999.0
1010,2024-08-09,Priya Iyer,Hyderabad,Office Chair,Furniture,2,12998.0


In [0]:
# ---------- View 2: vw_product_sales_agg ----------

create_or_replace_view(
    "vw_product_sales_agg",
    f"""
        SELECT
            p.product_id, p.product_name,
            SUM(o.quantity) AS total_quantity
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.products p ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
    """,
    "v1: initial aggregation - total quantity sold per product",
    changed_at_override="2024-02-01 11:00:00",
)

create_or_replace_view(
    "vw_product_sales_agg",
    f"""
        SELECT
            p.product_id, p.product_name, p.category,
            SUM(o.quantity) AS total_quantity,
            ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.products p ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name, p.category
    """,
    "v2: added category and total_revenue metric",
    changed_at_override="2024-05-10 16:45:00",
)

create_or_replace_view(
    "vw_product_sales_agg",
    f"""
        SELECT
            p.category,
            p.product_id, p.product_name,
            SUM(o.quantity) AS total_quantity,
            ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue,
            RANK() OVER (PARTITION BY p.category ORDER BY SUM(o.quantity * p.unit_price) DESC) AS revenue_rank_in_category
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.products p ON o.product_id = p.product_id
        GROUP BY p.category, p.product_id, p.product_name
    """,
    "v3: added category-level revenue ranking window function",
    changed_at_override="2024-09-05 13:20:00",
)

display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.vw_product_sales_agg ORDER BY category, revenue_rank_in_category"))

[v1: initial aggregation - total quantity sold per product] -> workspace.view_lineage_demo.vw_product_sales_agg
[v2: added category and total_revenue metric] -> workspace.view_lineage_demo.vw_product_sales_agg
[v3: added category-level revenue ranking window function] -> workspace.view_lineage_demo.vw_product_sales_agg


category,product_id,product_name,total_quantity,total_revenue,revenue_rank_in_category
Electronics,102,Mechanical Keyboard,3,8997.0,1
Electronics,101,Wireless Mouse,5,3995.0,2
Furniture,104,Standing Desk,2,31998.0,1
Furniture,103,Office Chair,3,19497.0,2
Stationery,105,Notebook Set,15,3735.0,1


## 5. Sample downstream operations (aggregations on top of the views)

In [0]:
display(spark.sql(f"""
    SELECT city, COUNT(DISTINCT order_id) AS orders, SUM(total_amount) AS revenue
    FROM {CATALOG}.{SCHEMA}.vw_customer_order_summary
    GROUP BY city
    ORDER BY revenue DESC
"""))

display(spark.sql(f"""
    SELECT category, SUM(total_revenue) AS category_revenue
    FROM {CATALOG}.{SCHEMA}.vw_product_sales_agg
    GROUP BY category
    ORDER BY category_revenue DESC
"""))

city,orders,revenue
Bengaluru,2,22498.0
Chennai,4,21239.0
Hyderabad,2,18996.0
Mumbai,2,5489.0


category,category_revenue
Furniture,51495.0
Electronics,12992.0
Stationery,3735.0


## 6. Extract metadata

### 6a. View metadata (from `information_schema.views`)

In [0]:
views_metadata_df = spark.sql(f"""
    SELECT
        table_catalog AS view_catalog,
        table_schema  AS view_schema,
        table_name    AS view_name,
        view_definition
    FROM {CATALOG}.information_schema.views
    WHERE table_schema = '{SCHEMA}'
""")
display(views_metadata_df)

view_catalog,view_schema,view_name,view_definition
workspace,view_lineage_demo,vw_product_sales_agg,"SELECT p.category, p.product_id, p.product_name, SUM(o.quantity) AS total_quantity, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue, RANK() OVER (PARTITION BY p.category ORDER BY SUM(o.quantity * p.unit_price) DESC) AS revenue_rank_in_category FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id GROUP BY p.category, p.product_id, p.product_name"
workspace,view_lineage_demo,vw_customer_order_summary,"SELECT o.order_id, o.order_date, c.customer_name, c.city, p.product_name, p.category, o.quantity, ROUND(o.quantity * p.unit_price, 2) AS total_amount FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id WHERE o.order_date >= DATE'2024-01-01'"


### 6b. View lineage — which table(s) each view is built from (`information_schema.view_table_usage`, Unity Catalog only)

In [0]:
if USE_UC:
    try:
        lineage_df = spark.sql(f"""
            SELECT
                view_catalog, view_schema, view_name,
                table_catalog AS source_table_catalog,
                table_schema  AS source_table_schema,
                table_name    AS source_table_name
            FROM {CATALOG}.information_schema.view_table_usage
            WHERE view_schema = '{SCHEMA}'
        """)
    except Exception as e:
        print(f"view_table_usage not available ({e}); returning empty lineage frame.")
        lineage_df = spark.sql("SELECT NULL AS view_catalog LIMIT 0")
else:
    print("Not on Unity Catalog -> view_table_usage is not available. Skipping lineage extraction.")
    lineage_df = spark.sql("SELECT NULL AS view_catalog LIMIT 0")

display(lineage_df)

view_table_usage not available ([TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`information_schema`.`view_table_usage` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`view_lineage_demo`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 7 pos 17;
'Project ['view_catalog, 'view_schema, 'view_name, 'table_catalog AS source_table_catalog#14080, 'table_schema AS source_table_schema#14081, 'table_name AS source_table_name#14082]
+- 'Filter ('view_schema = view_lineage_demo)
   +- 'UnresolvedRelation [workspace, information_schema, view_table_usage], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.errors.QueryCompilationEr

view_catalog


### 6c. Change history — from our own `view_change_log` table (always available)

In [0]:
change_log_df = spark.sql(f"""
    SELECT view_full_name, change_description, changed_by, changed_at, definition_sql
    FROM {CATALOG}.{SCHEMA}.view_change_log
    ORDER BY view_full_name, changed_at
""")
display(change_log_df)

view_full_name,change_description,changed_by,changed_at,definition_sql
workspace.view_lineage_demo.vw_customer_order_summary,"v1: initial join of orders, customers, products",thrishok26@gmail.com,2024-01-15T10:00:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, p.product_name, o.quantity FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id"
workspace.view_lineage_demo.vw_customer_order_summary,"v1: initial join of orders, customers, products",thrishok26@gmail.com,2024-01-15T10:00:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, p.product_name, o.quantity FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id"
workspace.view_lineage_demo.vw_customer_order_summary,"v2: added customer city, product category and computed total_amount",thrishok26@gmail.com,2024-04-02T14:30:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, c.city, p.product_name, p.category, o.quantity, ROUND(o.quantity * p.unit_price, 2) AS total_amount FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id"
workspace.view_lineage_demo.vw_customer_order_summary,"v2: added customer city, product category and computed total_amount",thrishok26@gmail.com,2024-04-02T14:30:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, c.city, p.product_name, p.category, o.quantity, ROUND(o.quantity * p.unit_price, 2) AS total_amount FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id"
workspace.view_lineage_demo.vw_customer_order_summary,v3: added filter to restrict to orders from 2024 onward,thrishok26@gmail.com,2024-08-20T09:15:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, c.city, p.product_name, p.category, o.quantity, ROUND(o.quantity * p.unit_price, 2) AS total_amount FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id WHERE o.order_date >= DATE2024-01-01"
workspace.view_lineage_demo.vw_customer_order_summary,v3: added filter to restrict to orders from 2024 onward,thrishok26@gmail.com,2024-08-20T09:15:00.000Z,"SELECT o.order_id, o.order_date, c.customer_name, c.city, p.product_name, p.category, o.quantity, ROUND(o.quantity * p.unit_price, 2) AS total_amount FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.customers c ON o.customer_id = c.customer_id JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id WHERE o.order_date >= DATE2024-01-01"
workspace.view_lineage_demo.vw_product_sales_agg,v1: initial aggregation - total quantity sold per product,thrishok26@gmail.com,2024-02-01T11:00:00.000Z,"SELECT p.product_id, p.product_name, SUM(o.quantity) AS total_quantity FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id GROUP BY p.product_id, p.product_name"
workspace.view_lineage_demo.vw_product_sales_agg,v1: initial aggregation - total quantity sold per product,thrishok26@gmail.com,2024-02-01T11:00:00.000Z,"SELECT p.product_id, p.product_name, SUM(o.quantity) AS total_quantity FROM workspace.view_lineage_demo.orders o JOIN workspace.view_lineage_demo.products p ON o.product_id = p.product_id GROUP BY p.product_id, p.product_name"
workspace.view_lineage_demo.vw_product_sales_agg,v2: added category and total_revenue metric,thrishok26@gmail.com,2024-05-10T16:45:00.000Z,"SELECT p.product_id, p.product_name, p.category, SUM(o.quantity) 

### 6d. (Optional) Change history — real audit events from `system.access.audit`

Only works if:
- Unity Catalog **system tables** are enabled by a workspace/account admin, and
- you have `SELECT` permission on `system.access.audit`.

Wrapped in try/except so the notebook never breaks if this isn't available.

In [0]:
try:
    audit_df = spark.sql(f"""
        SELECT
            event_date, event_time, user_identity.email AS performed_by,
            action_name, request_params
        FROM system.access.audit
        WHERE service_name = 'unityCatalog'
          AND action_name IN ('createView', 'alterView', 'createOrReplaceView')
          AND request_params.full_name_arg LIKE '%{SCHEMA}%'
        ORDER BY event_time
    """)
    audit_available = True
    display(audit_df)
except Exception as e:
    print(f"system.access.audit not available in this workspace ({e}).")
    print("Skipping — relying on the custom view_change_log table instead.")
    audit_available = False
    audit_df = None

event_date,event_time,performed_by,action_name,request_params


## 7. Write everything to an Excel file

In [0]:
%pip install -q openpyxl

import pandas as pd

tables_overview_pdf = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").toPandas()
views_metadata_pdf  = views_metadata_df.toPandas()
lineage_pdf         = lineage_df.toPandas()
change_log_pdf      = change_log_df.toPandas()

# make timestamps timezone-naive so Excel writer doesn't choke on them
for pdf in (views_metadata_pdf, change_log_pdf):
    for col in pdf.columns:
        if pd.api.types.is_datetime64_any_dtype(pdf[col]):
            pdf[col] = pdf[col].dt.tz_localize(None) if pdf[col].dt.tz is not None else pdf[col]

local_path = "/tmp/view_metadata_report.xlsx"

with pd.ExcelWriter(local_path, engine="openpyxl") as writer:
    tables_overview_pdf.to_excel(writer, sheet_name="Tables_Overview", index=False)
    views_metadata_pdf.to_excel(writer, sheet_name="Views_Metadata", index=False)
    lineage_pdf.to_excel(writer, sheet_name="View_Lineage", index=False)
    change_log_pdf.to_excel(writer, sheet_name="View_Change_History", index=False)
    if audit_available and audit_df is not None:
        audit_df.toPandas().to_excel(writer, sheet_name="Audit_Log_Realtime", index=False)

print(f"Excel report written locally to: {local_path}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Excel report written locally to: /tmp/view_metadata_report.xlsx


## 8. Move the Excel file to a persistent/shareable location

Pick whichever matches your workspace:
- **Unity Catalog Volume** (recommended if you have one) — uncomment and set the path.
- **DBFS FileStore** — works on most workspaces where DBFS root access is enabled.

In [0]:
dbfs_path = "dbfs:/FileStore/view_lineage_demo/view_metadata_report.xlsx"

try:
    dbutils.fs.mkdirs("dbfs:/FileStore/view_lineage_demo")
    dbutils.fs.cp(f"file:{local_path}", dbfs_path)
    print(f"Copied to {dbfs_path}")
    print("Download via: https://<your-workspace-host>/files/view_lineage_demo/view_metadata_report.xlsx")
except Exception as e:
    print(f"Could not copy to DBFS FileStore ({e}).")

# --- Uncomment + adjust if you have a Unity Catalog Volume available ---
# volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/<your_volume_name>/view_metadata_report.xlsx"
# dbutils.fs.cp(f"file:{local_path}", volume_path)
# print(f"Copied to {volume_path}")

Could not copy to DBFS FileStore ([DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /FileStore/view_lineage_demo SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.mkdirs(DatabricksFileSystemV2.scala:747)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.mkdirs(DatabricksFileSystem.scala:209)
	at com.databricks.sql.io.LokiFileSystem.mkdirs(LokiFileSystem.scala:618)
	at org.apache.hadoop.fs.FileSystem.mkdirs(FileSystem.java:2496)
	at com.databricks.backend.daemon.dbutils.FSUtils.$anonfun$mkdirs$3(DBUtilsCore.scala:627)
	at com.databricks.backend.daemon.dbutils.FSUtils.withFsSafetyCheck(DBUtilsCore.scala:209)
	at com.databricks.backend.daemon.dbutils.FSUtils.$anonfun$mkdirs$2(DBUtilsCore.scala:625)
	at com.databricks

## Notes for adapting this to the real client setup

- Point `CATALOG` / `SCHEMA` widgets at the client's real catalog/schema — sections 6a/6b work
  unchanged against any existing Unity Catalog views, no need to recreate the demo tables/views.
- For genuine historical tracking of view changes going forward, wire `create_or_replace_view()`
  (or an equivalent logging step) into whatever deploys/alters views in the client's pipeline
  (dbt hook, Databricks Asset Bundle job, CI/CD step, etc.) so every future change is captured.
- If system tables (`system.access.audit`) are enabled for the client's account, section 6d gives
  you the *actual* historical `createView`/`alterView` events without needing your own logging.